# 第9周 · 周三：DID 实现与检验——验证你的假设\n\n> 难度：⭐⭐⭐⭐ | 定位：掌握DID的完整实现流程，重点在平行趋势检验和安慰剂检验\n> 前置依赖：周二DID原理 | 后续用途：W10-W12综合案例中DID部分的完整实现\n> 面试地位：能否做平行趋势检验是区分"会用DID"和"真正理解DID"的分水岭\n\n---\n\n## 📘 区块1：核心概念与第一性原理\n\n### 1.1 DID的实现不仅仅是跑一个回归\n\n很多人在工作中"做了DID"——跑了一个OLS回归，看到交互项系数显著，就写报告说"处理有效果"。但真正的DID分析包含四个紧密相连的步骤，缺一个都不能算"完成了DID"：\n\n```\n步骤1：描述性分析——画趋势图，眼看为实\n步骤2：基准回归——跑DID回归，得到初步估计\n步骤3：平行趋势检验——事件研究图，检验核心假设\n步骤4：稳健性检验——安慰剂检验、替换对照组、替换变量定义\n```\n\n这四个步骤中，最重要的是步骤3——如果平行趋势不成立，步骤2的系数再显著也没有意义。\n\n### 1.2 事件研究法(Event Study)的本质\n\n事件研究法是DID的"动态版本"。它不假设处理效应在处理后每一期都相同，而是允许效应随时间变化。更重要的是，它在处理前也有系数——这才是检验平行趋势的关键。\n\n**事件研究回归方程**：\n```\nY_it = α_i + λ_t + Σ_{k≠-1} β_k · D_{it}^k + ε_it\n```\n\n其中 $D_{it}^k$ 是一组"相对时间"的虚拟变量：\n- k = -4, -3, -2：处理前各期\n- k = 0, 1, 2, 3：处理当期+处理后各期\n- 通常以k=-1（处理前最后一期）作为参照组\n\n**解读规则**：\n- k < 0的系数应接近0（不显著）→ 平行趋势成立\n- k ≥ 0的系数反映每期的处理效应 → 可以看到效应的"动态路径"\n- 如果k < 0的系数显著偏离0 → 平行趋势不成立 → DID不可信\n\n### 1.3 安慰剂检验(Placebo Test)的两种形式\n\n**形式1：时间安慰剂（In-Time Placebo）**\n\n把"处理时间"提前到实际处理之前（如提前2期），用处理前的数据重新跑DID。\n如果交互项仍然"显著"→ 说明处理前两组已经不平行了→DID不可信。\n如果交互项不显著→ 增强平行趋势的可信度。\n\n**形式2：组别安慰剂（In-Space Placebo）**\n\n随机打乱"谁是处理组谁是对照组"，重新跑DID，重复1000次。画出1000个"假DID估计量"的分布，看真实DID估计量在这个分布中的位置。\n如果真实DID落在这个分布的"正常范围"内（如95%以内）→ 可能只是随机波动。\n如果真实DID在这个分布的尾巴上（如远超95%分位数）→ 效果不太可能是偶然的。\n\n### 1.4 面试高频考点\n\n**面试官常问**：\n- "你做了一个DID，怎么说服我相信平行趋势成立？"\n- "事件研究图中，处理前有一个系数显著了，这说明什么？你怎么办？"\n- "安慰剂检验怎么做？为什么它不能'证明'平行趋势？"\n- "聚类标准误应该在哪个级别聚类？为什么？"\n\n**回答要点**：\n- 画事件研究图+处理前系数联合检验（F test: 所有处理前系数=0）\n- 一个处理前系数显著可能只是多重检验的假阳性→看整体趋势（联合检验）\n- 时间安慰剂=重新定义处理时间；组别安慰剂=随机打乱处理组；安慰剂检验增强可信度但不能"证明"\n- 在"处理分配的级别"聚类（如城市级别的政策→聚类到城市；用户级别的AB→聚类到用户）

## 🧩 区块2：知识点详细拆解\n\n### 2.1 事件研究图的绘制与解读\n\n**为什么重要**：事件研究图是DID分析中"第一重要的图"——比任何数字都直观、都有说服力。\n\n**绘制步骤**：\n1. 生成相对时间变量（relative_time = 实际时间 - 处理时间）\n2. 创建一组相对时间的虚拟变量（或以-1期为基准）\n3. 用TWFE回归估计每个相对时间点的系数\n4. 画系数和置信区间的点线图\n\n**好的事件研究图长什么样**：\n- 横轴：相对时间（-4,-3,-2,-1,0,+1,+2,+3）\n- 纵轴：各期的估计系数和95%CI\n- 有一条x=0的垂直虚线（处理开始的时刻）\n- 处理前各期的系数接近0，置信区间包含0\n- 处理后各期的系数"跳起来"并持续为正（或负）\n\n**常见的图解读错误**：\n- "处理前p>0.05，所以平行趋势成立"→ 单个系数不显著≠平行趋势成立。正确做法是联合检验（F test: 所有处理前系数同时=0）\n- "处理前系数看起来都接近0"→ "看起来"不够，需要统计检验。但统计检验又不保险（在样本量小的时候检验力不足）→ 这是一个权衡\n\n### 2.2 聚类标准误的选择\n\n**为什么重要**：标准误的正确选择直接影响你的结论——是"显著"还是"不显著"。\n\n**DID中的聚类层级选择**：\n\n| 处理分配的级别 | 聚类级别 | 例子 |\n|-------------|---------|------|\n| 城市级 | 城市 | 某市推出新政策→聚类到城市 |\n| 学校级 | 学校 | 某校推出新教学方法→聚类到学校 |\n| 用户级 | 用户 | 给部分用户发了优惠券→聚类到用户 |\n| 州/省级 | 州/省 | 某省推出新法规→聚类到省 |\n\n**核心原则**：在**处理分配的级别**聚类。如果处理是针对城市的，标准误应该在城市级别聚类——因为同一个城市不同时期的误差是相关的。\n\n**常见问题**：\n- "聚类数量太少(<20)怎么办？"→ 传统聚类标准误在小聚类数下偏小（假阳性增多）。考虑：wild bootstrap、随机化推断、或使用偏误修正的聚类标准误\n- "混合截面数据需要聚类吗？"→ 如果是面板数据（同一个体多期）→必须聚类。如果是重复截面（每期都是新的样本）→一般不需要在个体级别聚类，但可能需要在更高层级聚类\n\n### 2.3 固定效应 vs 一阶差分\n\n**为什么重要**：两种估计方法在特定条件下等价，但选择影响推断。\n\n**固定效应模型(TWFE)**：\n```\nY_it = α_i + λ_t + β·D_it + ε_it\n```\n- 等价于对"去均值后的数据"做回归\n- 用OLS标准误时不正确（需要聚类）\n\n**一阶差分模型(FD)**：\n```\nΔY_it = Δλ_t + β·ΔD_it + Δε_it\n```\n- 在只有2期时，FD和FE等价\n- 在多于2期时，如果ε_it是随机游走→FD更有效；如果ε_it是iid→FE更有效\n\n**选择准则**：大多数情况用FE（效率更高）。如果怀疑误差有序列相关性（DW检验显著），考虑FD。\n\n### 2.4 处理效应异质性——DID估计的"隐藏杀手"\n\n**为什么重要**：如果处理效应在不同时间/不同个体上不同，TWFE估计的β可能不是你想估计的ATT。\n\n**问题举例**：\n\n某APP在1月给安卓用户上线了新功能（处理组），3月给iOS用户上线（也是处理组但晚一些）。你想估计这个功能的效果。\n\n如果处理效应随时间递减（如第一个月效果很强，之后减弱），传统TWFE的估计量将是：\n- 早期处理组(安卓)的效应（已衰减）\n- 后期处理组(iOS)的效应（刚上线，还很强）\n- 以及一些"已处理vs未处理"和"已处理vs已处理"的比较\n- → 这个加权平均可能和你想要的任何一个ATT都不一样！\n\n**解决的三个方向**：\n1. 用Sun & Abraham(2021)或Callaway & Sant'Anna(2021)的异质性稳健估计量\n2. 分别估计每个cohort的处理效应，然后做描述性对比\n3. 用事件研究法（默认允许每期效应不同）\n\n### 2.5 匹配+DID（PSM-DID）\n\n**为什么重要**：当处理组和对照组在可观测特征上差异很大时，先用PSM匹配、再做DID，可以增强平行趋势的可信度。\n\n**流程**：\n1. 用处理前的协变量估计倾向得分\n2. 在处理前进行匹配（不能在处理后匹配——会导致"匹配未来"的偏误）\n3. 只保留匹配上的样本\n4. 在匹配后的样本上做DID\n\n**PSM-DID的优势**：\n- 比纯PSM强：消除了不随时间变化的未观测混淆（DID的优势）\n- 比纯DID强：两组在处理前的特征更平衡（PSM的优势）\n- 是实证研究中最常见的"组合拳"之一\n\n### 2.6 灵敏度分析——"多少混淆才够推翻我的结论"\n\n**为什么重要**：平行趋势假设不可检验，灵敏度分析可以回答"如果平行趋势不成立，偏差多大才改变结论"。\n\n**Rambachan & Roth (2023)方法**：\n不假设"平行趋势严格成立"，而是假设"处理前趋势的偏离程度不会超过某个M"。然后问：在M的约束下，处理后1期/2期的真实效应可能在什么范围？\n\n**实际操作**：\n1. 估计事件研究模型，得到处理前各期的系数\n2. 从处理前系数的变异性，推断"趋势偏离的合理程度"\n3. 构建"对平行趋势偏离部分稳健"的置信区间\n4. 报告：即使允许平行趋势有一定程度的偏离，结论仍然稳健（或不稳健）\n\n### 2.7 DID分析清单——一个都不能少\n\n**为什么重要**：论文和报告中DID分析的"最少操作集"。\n\n| 步骤 | 内容 | 必须吗？ |\n|------|------|---------|\n| 1. 描述性图 | 画处理组和对照组的均值趋势图 | 必须 |\n| 2. 基准回归 | 2×2 DID或TWFE | 必须 |\n| 3. 事件研究图 | 处理前后各期系数+CI | 必须 |\n| 4. 联合检验 | F test: 所有处理前系数=0 | 必须 |\n| 5. 时间安慰剂 | 把处理时间提前 | 建议 |\n| 6. 组别安慰剂 | 随机打乱处理组 | 建议 |\n| 7. 替换对照组 | 换一个对照组/变量的定义 | 建议 |\n| 8. 灵敏度分析 | 平行趋势偏离下的稳健性 | 前沿 |\n| 9. 聚类标准误 | 处理分配级别聚类 | 必须 |

## 💻 区块3：动手练习\n\n### 步骤0：环境准备

In [ ]:
import numpy as np; import pandas as pd\nimport matplotlib.pyplot as plt\nimport statsmodels.formula.api as smf\nfrom scipy import stats\nimport warnings; warnings.filterwarnings('ignore')\nnp.random.seed(42)\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']\nplt.rcParams['axes.unicode_minus'] = False\nprint('='*60)\nprint('DID实现与检验——动手练习')\nprint('='*60)

### 步骤1：事件研究法——平行趋势检验\n\n1.1 生成多期面板数据\n1.2 做事件研究回归，画系数图

In [ ]:
# 步骤1.1：生成多期面板数据\nprint('【步骤1.1】生成多期面板数据（10个城市×9期）')\nprint('-'*50)\n\nn = 10\nperiods = list(range(-4, 5))  # 相对时间-4到+4\ntreated = np.array([1]*5 + [0]*5)\ncity_base = np.random.normal(100, 10, n)\ntreat_effect = 8  # 真实处理效应\n\ndata = []\nfor i in range(n):\n    for period in periods:\n        time_trend = 2 * period\n        effect_active = 1 if period >= 0 else 0\n        effect = treat_effect * treated[i] * effect_active\n        y = city_base[i] + time_trend + effect + np.random.normal(0, 3)\n        data.append({'city': i, 'period': period, 'treated': treated[i], 'y': y})\ndf = pd.DataFrame(data)\n\n# 创建相对时间的虚拟变量（以period=-1为基准）\nfor p in periods:\n    if p != -1:\n        df[f'rel_time_{p}'] = (df['period'] == p).astype(int)\n        df[f'treat_rel_{p}'] = df[f'rel_time_{p}'] * df['treated']\n\nprint(f'数据规模: {len(df)}行 ({n}城市 × {len(periods)}期)')\nprint(f'处理组城市: {treated.sum()}个, 对照组城市: {(1-treated).sum()}个')\nprint(f'时期范围: {periods[0]}~{periods[-1]} (0=处理开始)')

In [ ]:
# 步骤1.2：事件研究回归\nprint('\\n【步骤1.2】事件研究法——估计每个相对时间点的效应')\nprint('-'*50)\n\n# 构建回归公式：y ~ 各期交互项 + 城市固定效应 + 时间固定效应\ntreat_rel_cols = [f'treat_rel_{p}' for p in periods if p != -1]\ncity_dummies = ' + '.join([f'C(city)[T.{i}]' for i in range(1, n)])\nperiod_dummies = ' + '.join([f'C(period)[T.{p}]' for p in periods if p != -1])\n\nformula = f'y ~ {" + ".join(treat_rel_cols)} + {city_dummies} + {period_dummies}'\nmodel = smf.ols(formula, data=df).fit()\n\n# 提取系数（交互项代表该期处理效应）\ncoefs = {}\ncis = {}\nfor p in periods:\n    if p == -1:\n        coefs[p] = 0; cis[p] = (0, 0)  # 基准期\n    else:\n        col = f'treat_rel_{p}'\n        if col in model.params.index:\n            coefs[p] = model.params[col]\n            ci = model.conf_int().loc[col]\n            cis[p] = (ci[0], ci[1])\n\n# 画事件研究图\nfig, ax = plt.subplots(figsize=(10, 5))\nsorted_periods = sorted(coefs.keys())\nestimates = [coefs[p] for p in sorted_periods]\nlower = [coefs[p] - cis[p][0] for p in sorted_periods]\nupper = [cis[p][1] - coefs[p] for p in sorted_periods]\n\nax.errorbar(sorted_periods, estimates, \n           yerr=[lower, upper], fmt='o', capsize=4, capthick=1.5,\n           markersize=8, linewidth=2, color='darkblue')\nax.axvline(x=-1, color='red', linestyle='--', linewidth=1.5, label='处理开始(period=0)')\nax.axhline(y=0, color='gray', linewidth=0.8)\nax.set_xlabel('相对时间(期)'); ax.set_ylabel('处理效应估计量')\nax.set_title('事件研究图：平行趋势检验')\nax.legend()\nplt.tight_layout(); plt.show()\n\nprint('\\n解读：')\nprint('  处理前(p<0)系数: ', end='')\nfor p in sorted_periods:\n    if p < 0:\n        print(f'{coefs[p]:+.2f}  ', end='')\nprint('→ 接近0 → 平行趋势成立 ✓')\nprint('  处理后(p>=0)系数: ', end='')\nfor p in sorted_periods:\n    if p >= 0:\n        print(f'{coefs[p]:+.2f}  ', end='')\nprint(f'→ 接近真实效应{treat_effect}')

### 步骤2：安慰剂检验\n\n2.1 时间安慰剂——把处理时间提前\n2.2 组别安慰剂——随机打乱处理组

In [ ]:
# 步骤2.1：时间安慰剂检验——把"处理时间"提前到period=-2\nprint('\\n【步骤2.1】时间安慰剂：假设处理发生在period=-2（实际之前）')\nprint('-'*50)\n\ndf['fake_post'] = (df['period'] >= -2).astype(int)  # 假装处理时间在-2\ndf['treated_fake_post'] = df['treated'] * df['fake_post']\n\n# 只用处理前的数据（period < 0）做这个检验\ndf_pre = df[df['period'] < 0].copy()\nmodel_placebo = smf.ols('y ~ treated + fake_post + treated_fake_post', data=df_pre).fit()\nplacebo_coef = model_placebo.params['treated_fake_post']\nplacebo_p = model_placebo.pvalues['treated_fake_post']\n\nprint(f'时间安慰剂DID估计: {placebo_coef:.3f} (p={placebo_p:.3f})')\nif placebo_p > 0.05:\n    print('  → 不显著 ✓ 增强平行趋势可信度')\nelse:\n    print('  → 显著！⚠ 处理前两组已经不平行了！DID不可信')\n\n# 步骤2.2：组别安慰剂——随机打乱1000次\nprint('\\n【步骤2.2】组别安慰剂：随机打乱处理组1000次')\nprint('-'*50)\n\ntrue_did = did_manual if 'did_manual' in dir() else -8  # 从步骤1获取\n# 重新用2×2 DID格式\ndf_22 = df[(df['period'].isin([-1, 0]))].copy()  # 取前1期和后1期\ndf_22['post'] = (df_22['period'] >= 0).astype(int)\ndf_22['treated_post'] = df_22['treated'] * df_22['post']\nmodel_true = smf.ols('y ~ treated + post + treated_post', data=df_22).fit()\ntrue_did = model_true.params['treated_post']\n\nn_permutations = 1000\nplacebo_dids = []\nfor _ in range(n_permutations):\n    df_perm = df_22.copy()\n    # 随机打乱treated标签（在city级别！）\n    cities = df_perm['city'].unique()\n    np.random.shuffle(treated)\n    city_to_treat = {c: treated[i] for i, c in enumerate(cities)}\n    df_perm['treated'] = df_perm['city'].map(city_to_treat)\n    df_perm['treated_post'] = df_perm['treated'] * df_perm['post']\n    model_perm = smf.ols('y ~ treated + post + treated_post', data=df_perm).fit()\n    placebo_dids.append(model_perm.params['treated_post'])\n\nplacebo_dids = np.array(placebo_dids)\np_value_perm = np.mean(np.abs(placebo_dids) >= np.abs(true_did))\n\nfig, ax = plt.subplots(figsize=(10, 5))\nax.hist(placebo_dids, bins=40, color='lightblue', edgecolor='darkblue', alpha=0.7)\nax.axvline(x=true_did, color='red', linewidth=2, linestyle='--', label=f'真实DID={true_did:.1f}')\nax.set_xlabel('DID估计量'); ax.set_ylabel('频数(1000次随机打乱)')\nax.set_title(f'组别安慰剂检验\\np值(经验)={p_value_perm:.3f}')\nax.legend()\nplt.tight_layout(); plt.show()\n\nprint(f'真实DID: {true_did:.1f}')\nprint(f'安慰剂分布均值: {np.mean(placebo_dids):.3f}, SD: {np.std(placebo_dids):.3f}')\nprint(f'经验p值: {p_value_perm:.3f}')\nif p_value_perm < 0.05:\n    print('  → 真实DID在安慰剂分布的尾巴上 → 不太可能是偶然 ✓')\nelse:\n    print('  → 真实DID在安慰剂分布的正常范围内 → 可能是偶然 ⚠')

### 步骤3：挑战练习——完整DID分析流程\n\n3.1 模拟一个真实的交错处理场景\n3.2 对比传统TWFE和事件研究法的结果差异

In [ ]:
# 步骤3：完整DID分析——交错处理(Staggered Adoption)场景\nprint('\\n【步骤3】交错处理DID——不同城市在不同时间推出政策')\nprint('='*60)\n\nnp.random.seed(2024)\nn_cities = 30\nn_periods = 12\n\n# 每个城市在不同的时期"被处理"（或始终不被处理）\ntreat_time = np.random.choice([3, 5, 7, 999], n_cities, p=[0.3, 0.3, 0.2, 0.2])\n# treat_time=999 → 从未被处理（始终对照组）\n\ncity_base = np.random.normal(100, 15, n_cities)\n\n# 生成数据\ndata_stag = []\nfor i in range(n_cities):\n    for t in range(n_periods):\n        treated_now = 1 if t >= treat_time[i] else 0\n        time_trend = 1.5 * t\n        effect = 10 if treated_now == 1 else 0  # 处理效应=10\n        y = city_base[i] + time_trend + effect + np.random.normal(0, 5)\n        data_stag.append({'city': i, 'time': t, 'treated': treated_now, 'y': y, 'treat_time': treat_time[i]})\ndf_stag = pd.DataFrame(data_stag)\n\n# 方法1：传统TWFE\ndf_stag['city_cat'] = df_stag['city'].astype('category')\ndf_stag['time_cat'] = df_stag['time'].astype('category')\nmodel_twfe = smf.ols('y ~ treated + C(city) + C(time)', data=df_stag).fit()\ntwfe_est = model_twfe.params['treated']\nprint(f'传统TWFE估计: {twfe_est:.2f} (真实效应=10)')\n\n# 方法2：事件研究法——更灵活，允许效应随时间变化\ndf_stag['rel_time'] = df_stag['time'] - df_stag['treat_time']\ndf_stag['rel_time'] = df_stag['rel_time'].clip(-4, 4)  # 截断到[-4,4]\n\n# 创建相对时间虚拟变量\nfor rt in range(-4, 5):\n    if rt != -1:\n        df_stag[f'rel_{rt}'] = (df_stag['rel_time'] == rt).astype(int)\n\nrel_cols = [f'rel_{rt}' for rt in range(-4, 5) if rt != -1]\nformula_es = f'y ~ {" + ".join(rel_cols)} + C(city) + C(time)'\nmodel_es = smf.ols(formula_es, data=df_stag).fit()\n\n# 提取事件研究系数\nes_coefs = {}\nfor rt in range(-4, 5):\n    if rt == -1:\n        es_coefs[rt] = 0\n    else:\n        col = f'rel_{rt}'\n        if col in model_es.params.index:\n            es_coefs[rt] = model_es.params[col]\n        else:\n            es_coefs[rt] = np.nan\n\n# 画两种方法的对比\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# TWFE\naxes[0].bar(['TWFE估计'], [twfe_est], color='steelblue')\naxes[0].axhline(y=10, color='red', linestyle='--', label='真实效应=10')\naxes[0].set_title(f'TWFE: {twfe_est:.1f}')\naxes[0].legend()\n\n# 事件研究\nrt_sorted = sorted([k for k in es_coefs.keys() if not np.isnan(es_coefs[k])])\ncoef_vals = [es_coefs[rt] for rt in rt_sorted]\naxes[1].plot(rt_sorted, coef_vals, 'o-', linewidth=2, markersize=8)\naxes[1].axvline(x=-0.5, color='red', linestyle='--')\naxes[1].axhline(y=0, color='gray', linewidth=0.8)\naxes[1].set_xlabel('相对处理时间'); axes[1].set_ylabel('估计系数')\naxes[1].set_title('事件研究：每期效应估计')\nplt.tight_layout(); plt.show()\n\nprint(f'\\n结论：')\nprint(f'  TWFE估计={twfe_est:.1f}（将所有时期和组别的比较加权平均）')\nprint(f'  事件研究法展示了效应的动态路径：处理前≈0，处理后≈10')\nprint(f'  在真实交错处理场景中，两种方法可能给出不同的估计——')\nprint(f'  这是因为TWFE的权重结构可能导致偏误（尤其是处理效应异质时）')

## 📊 区块4：业务场景应用\n\n### 场景1："某某政策带来了多少就业增长"——怎样做一个让人信服的DID\n\n**背景**：某省2024年6月推出了"大学生创业补贴"政策。省政府想知道这个政策到底带来了多少新增就业。你有该省和邻省(未推出此政策)的季度就业数据(2020Q1-2025Q4)。\n\n**完整分析方案**：\n\n1. **描述性分析（先说故事）**：画两条线——该省和邻省的就业趋势。标注"政策时间点"(2024Q2)。用肉眼判断：处理前两省趋势是否一致？处理后是否出现分化？\n\n2. **事件研究（核心证据）**：\n   - 横轴：相对政策时间的季度(-8到+6)\n   - 处理前季度(-8到-1)：系数在0附近小幅波动，联合检验p=0.35→平行趋势通过 ✓\n   - 处理后季度(0到+6)：系数从+0.5%(Q1)逐步上升到+3.2%(Q6)→政策效果有滞后性和累积性\n\n3. **安慰剂检验（增强说服力）**：\n   - 时间安慰剂：假设政策在2022Q2推出→DID不显著(p=0.68)\n   - 组别安慰剂：用邻省作为"假处理组"，本省作为"假对照组"→DID不显著\n\n4. **灵敏度分析（承认局限）**：\n   - 如果平行趋势有最多0.3%/季度的偏离，处理后6期的效果估计仍然为正(但不一定显著)\n   - → 结论：即使考虑一定的趋势偏离，政策效果仍然是正向的\n\n5. **结论**：政策在推出后6个季度内带来了约3%的额外就业增长。效果是渐进的——前期(0-2季度)效果微弱，中期(3-6季度)效果显著。\n\n### 场景2：DOU(日活)突增——是新功能带来的还是自然波动\n\n**背景**：5月10日上线了新版本的推荐算法，DOU从5月11日开始明显上升。PM发来贺报：新算法让DOU提升了12%。\n\n**你的反应（因果推断思维）**：\n\n1. **"提升12%"是跟什么比的？**\n   - 跟5月10日之前比？可能有周末效应/节日效应/投放效应\n   - 跟去年同期比？去年这个时候也可能有一个自然上升（暑假前）\n   - 需要一个对照组！\n\n2. **构造DID对照组**：\n   - 如果没有全量上线而是灰度：对比灰度和非灰度用户→直接是AB实验\n   - 如果全量上线：用"去年同期同时期"作为对照组→但要注意今年和去年可能有结构性变化\n   - 更好的方案：如果有一个"相似但不相关的指标"做对照。比如"推荐算法"只影响内容消费，不影响"工具功能使用"。用"工具功能DOU"作为对照做DID\n\n3. **DID分析**：\n   ```\n   内容DOU(处理): 上线前=100, 上线后=112, 变化=+12\n   工具DOU(对照): 上线前=100, 上线后=104, 变化=+4\n   DID = 12 - 4 = 8 → 算法真实提升约8%，不是12%\n   ```\n\n4. **向PM汇报**："算法的效果比12%更可能是约8%——因为同期还有4%的自然增长（所有功能都在涨）。建议再做一周观察，确定这个8%是否持续。"\n\n### 场景3：面试中拿DID案例展示分析能力\n\n**面试题**："请描述你用DID解决过的一个业务问题"\n\n**结构化回答模板**：\n\n1. **问题**："我们想评估会员体系改版是否带来了增量收入"\n2. **为什么不能直接对比**："改版后开通会员的用户特征和之前不同（选择偏差），不能直接对比开通前后的消费"\n3. **DID设计**：\n   - 处理组：在改版城市（北京/上海）的用户\n   - 对照组：未改版城市（广州/深圳）的用户\n   - 时间：改版前1个月 vs 改版后2个月\n4. **检验**："我先画了事件研究图，确认了处理前两组的消费趋势平行。做了组别安慰剂检验——随机打乱城市标签1000次，真实DID落到了分布的97.5%分位之外→效果是真实的"\n5. **结论**："DID估计会员改版带来了约15元/人/月的增量消费。按会员规模算，年化增量收入约2000万。但我也说明了局限——如果北京/上海和广州/深圳的用户在改版期间经历了不同的外部冲击（如地方消费券），估计可能有偏"

## ✅ 区块5：分级检验清单\n\n### 🟢 初级检验（必须能独立完成的）\n- [ ] 能画出事件研究图，并标注处理前后区域\n- [ ] 能写出事件研究回归的公式\n- [ ] 能做时间安慰剂检验（把处理时间提前）\n- [ ] 能解释为什么需要聚类标准误\n\n**答案提示**：\n- 横轴相对时间，纵轴系数+CI，x=0处画虚线\n- Y = α_i + λ_t + Σβ_k·D_k + ε，k是相对时间\n- 取处理前数据，重新定义"假处理时间"→交互项应不显著\n- 同一体多期残差相关→普通SE低估→聚类到处理分配级别\n\n### 🟡 中级检验（能解释给同事听的原理）\n- [ ] 能解释为什么事件研究优于简单的2×2 DID\n- [ ] 能判断聚类标准误应该在哪一级别聚类\n- [ ] 能说明交错处理下TWFE的偏误来源\n- [ ] 能解释组别安慰剂的原理和局限性\n\n**答案提示**：\n- 事件研究能看到动态路径+检验处理前平行趋势；2×2只看一个平均效应\n- 在"处理分配的级别"聚类（城市级政策→聚类到城市）\n- 已处理组作为"对照组"时有负权重→混合了不同cohort的不同效应\n- 随机打乱处理组1000次得到分布，看真实DID是否在尾部；局限：打乱可能破坏真实的数据结构\n\n### 🔴 高级检验（能举一反三的迁移能力）\n- [ ] 能设计一个包含事件研究+PSM-DID+安慰剂检验的完整DID分析方案\n- [ ] 能判断什么时候传统TWFE不安全（需要异质性稳健估计量）\n- [ ] 能进行Rambachan-Roth灵敏度分析\n- [ ] 能处理"处理前趋势本身就有差异"的场景（如加入特定趋势项/用DDD）\n\n**答案提示**：\n- 先用PSM匹配→事件研究检验→TWFE估计→时间+组别安慰剂→灵敏度分析\n- 当处理是交错的、且效应可能随时间变化时→TWFE不安全→用Sun&Abraham或CS方法\n- 灵敏度分析：允许处理前趋势有M的偏离→构建局部稳健置信区间\n- 加入城市特定线性趋势项(γ_i·t)可以部分缓解趋势不平行的问题;或用三重差分(DDD)再加一层对照

## 🔗 区块6：跨模块连接\n\n| 连接方向 | 关联内容 | 为什么有关 |\n|---------|---------|----------|\n| **← 前置依赖** | 周二DID原理 | 回归方程和估计量来自周二的推导 |\n| **← 前置依赖** | W6假设检验(F test/联合检验) | 平行趋势检验用F test验证所有处理前系数=0 |\n| **→ 后续用途** | W10-W12综合案例 | 综合案例中的DID部分需要完整的事件研究+安慰剂检验 |\n| **→ 后续用途** | 数据分析师的日常 | 任何策略评估类工作都需要这套检验流程 |\n| **业务连接** | 策略评估/政策评估/功能评估 | 所有"某事件前后的效果评估"的标准分析流程 |\n\n### 横向连接：与其他分析方法的关联\n\n**与假设检验的连接**：\n- 平行趋势的联合检验本质上是F检验（多元约束检验）\n- 安慰剂检验的p值本质上是随机化推断（Fisher's randomization test）\n\n**与可视化的连接**：\n- 事件研究图是因果推断中最重要的图表类型之一\n- 误差线(CI)的呈现直接影响读者的判断\n\n**与稳健性分析的连接**：\n- 替换变量/替换样本/替换方法→这是所有实证研究的标准操作\n- "结论不因建模选择而改变"是结论可信的标志

## 📝 区块7：当日小结\n\n> **DID的实现不仅仅是跑一个回归——完整的DID分析包含描述性图+基准回归+事件研究(平行趋势检验)+安慰剂检验+灵敏度分析。事件研究图是最重要的输出——它让你和读者同时看到三件事：处理前两组的趋势是否平行、处理效应的大小、效应的动态路径(是即时还是滞后的)。时间安慰剂("提前处理时间")和组别安慰剂("随机打乱处理组")不能"证明"平行趋势，但它们可以"证伪"你的DID——如果连安慰剂都"显著"了，说明你的DID结论不可靠。聚类标准误在DID中不是细节——它是"显著还是不显著"的决定性因素。交错处理下的传统TWFE可能产生严重有偏的估计——如果不同个体在不同时间接受处理，使用事件研究法或异质性稳健估计量(Sun&Abraham, Callaway&Sant'Anna)。**\n\n**三条金句带回家**：\n1. 事件研究图是DID分析中最重要的一张图——它同时回答了"假设成立吗"和"效果多大"两个问题\n2. 安慰剂检验不能"证明"平行趋势——但它可以揭示"平行趋势不成立"时你的DID会怎样\n3. 聚类标准误是必须的——你跑DID回归时的默认标准误大概率是错的（低估了）\n\n### 🔖 课后拓展\n\n1. **练习**：用你的数据跑一个事件研究回归，画系数图，判断平行趋势是否成立\n\n2. **思考**：如果你的事件研究图显示处理前3期的系数有一个"显著"了，你会怎么办？（提示：多重检验修正/联合检验/换对照组/换方法）\n\n3. **预习**：明天开始学习PSM——当DID不适用（没有自然对照组/没有处理前数据）时的替代方案\n\n4. **面试准备**：\n   - "你怎么说服我平行趋势成立？"\n   - "事件研究图中处理前有一个系数显著了，这说明什么？你怎么办？"\n   - "聚类标准误为什么是必须的？你应该在哪一级别聚类？"\n\n---\n\n*03analyst_thinking · Week 9 Day 3 · 2026-05-21*